# 11. Drug Profile Matrix & Model Selection Bridge

This notebook converts the complete exploratory data analysis (EDA Parts 1–9) into a quantitative **Drug Profile Matrix** for all 8 pharmaceutical categories (`M01AB`, `M01AE`, `N02BA`, `N02BE`, `N05B`, `N05C`, `R03`, `R06`).

This matrix serves as the **direct analytical bridge between empirical data characteristics and model selection strategy**.

---

### Key Profile Dimensions Analyzed:
1. **Trend**: Mean sales volume & 5-year growth/decline direction.
2. **Seasonality**: Peak month, minimum month, seasonal swing ratio ($	ext{Peak} / 	ext{Min}$), and seasonal strength $F_S$.
3. **Stationarity**: ADF & KPSS statistical tests on raw sales vs. Log1p first-differenced series ($\Delta \log(1+Y_t)$).
4. **Variance**: Heteroscedasticity ($\sigma_{	ext{raw}}$ vs $\sigma_{\log1p}$).
5. **ACF**: Autocorrelation persistence ($	ext{ACF}_1$, $	ext{ACF}_7$, $	ext{ACF}_{365}$).
6. **Notes / Bridge**: Strategic modeling implications per drug category.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

def get_data_path():
    candidates = [
        'dataset/saleshourly_preprocessed.csv',
        '../dataset/saleshourly_preprocessed.csv',
        'times_series/dataset/saleshourly_preprocessed.csv',
        '../times_series/dataset/saleshourly_preprocessed.csv'
    ]
    for cand in candidates:
        if os.path.exists(cand):
            return cand
    raise FileNotFoundError("Could not locate saleshourly_preprocessed.csv")

data_path = get_data_path()
print(f"Loading preprocessed dataset from: {data_path}")

df = pd.read_csv(data_path)
df['datetime'] = pd.to_datetime(df['datetime'])
df['date'] = df['datetime'].dt.date

drugs = ['M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06']
drug_descriptions = {
    'M01AB': 'Anti-inflammatory (Acetic acid)',
    'M01AE': 'Anti-inflammatory (Propionic acid)',
    'N02BA': 'Analgesics (Salicylic acid / Aspirin)',
    'N02BE': 'Analgesics (Pyrazolones / Paracetamol)',
    'N05B': 'Psycholeptics (Anxiolytics)',
    'N05C': 'Psycholeptics (Hypnotics / Sedatives)',
    'R03': 'Obstructive Airway Diseases (Asthma/COPD)',
    'R06': 'Antihistamines (Allergy)'
}

# Aggregate to Daily Series
daily_df = df.groupby('date')[drugs].sum()
daily_df.index = pd.to_datetime(daily_df.index)

print(f"Dataset Loaded Successfully: {len(daily_df):,} daily records ({daily_df.index.min().date()} to {daily_df.index.max().date()})")

Loading preprocessed dataset from: ../dataset/saleshourly_preprocessed.csv
Dataset Loaded Successfully: 2,106 daily records (2014-01-02 to 2019-10-08)


## Step 1: Compute Quantitative Profiling Metrics

In [2]:
profile_records = []

for d in drugs:
    s = daily_df[d]
    log_s = np.log1p(s)
    log_diff = log_s.diff().dropna()
    
    # 1. Trend Metrics
    mean_val = s.mean()
    start_avg = s.iloc[:90].mean()
    end_avg = s.iloc[-90:].mean()
    pct_change = ((end_avg - start_avg) / start_avg) * 100
    
    if pct_change > 15:
        trend_desc = f"Strong Upward (+{pct_change:.1f}%)"
    elif pct_change < -15:
        trend_desc = f"Downward Drift ({pct_change:.1f}%)"
    elif abs(pct_change) <= 15:
        trend_desc = f"Stable / Low Drift ({pct_change:+.1f}%)"
        
    # 2. Seasonality Metrics
    m_avg = df.groupby('Month')[d].mean()
    m_norm = m_avg / m_avg.mean()
    peak_m = m_norm.idxmax()
    min_m = m_norm.idxmin()
    peak_ratio = m_norm.max()
    min_ratio = m_norm.min()
    swing = peak_ratio / min_ratio
    
    month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
    seas_desc = f"{month_names[peak_m]} Peak ({peak_ratio:.2f}x) vs {month_names[min_m]} Min ({min_ratio:.2f}x) | Swing: {swing:.2f}x"
    
    # 3. Stationarity Tests
    adf_raw_p = adfuller(s)[1]
    kpss_raw_p = kpss(s, regression='c', nlags='auto')[1]
    adf_diff_p = adfuller(log_diff)[1]
    
    stat_desc = f"Raw ADF p={adf_raw_p:.4f} -> Diff ADF p={adf_diff_p:.1e} (Stationary d=1)"
    
    # 4. Variance & Noise
    raw_std = s.std()
    log_std = log_s.std()
    var_desc = f"Raw σ={raw_std:.2f} -> Log1p σ={log_std:.2f} (Log1p Stabilized)"
    
    # 5. ACF
    acf1 = s.autocorr(lag=1)
    acf7 = s.autocorr(lag=7)
    acf_desc = f"ACF1={acf1:.2f}, ACF7={acf7:.2f}"
    
    # 6. Strategic Model Notes
    if d in ['N02BE', 'R03']:
        notes = "High-Volume / Winter Peak. Requires Log1p + LightGBM / Prophet for dual 7d/365d seasonality."
    elif d == 'R06':
        notes = "Massive Summer Allergy Surge (May/Jun). Requires Dedicated Per-Drug Model or TFT Embeddings."
    elif d in ['M01AB', 'M01AE']:
        notes = "Noise-Dominated / Weak Seasonality. Regularized Ridge ML or LightGBM with short rolling features."
    elif d == 'N05B':
        notes = "Consistent Downward Trend. Models require differencing (d=1) or explicit negative trend slope."
    elif d == 'N05C':
        notes = "Low Volume Intermittent Sales. Quantile Regression (P10, P50, P90) for zero-inflation safety stock."
    else:
        notes = "Moderate Seasonality. SARIMAX or GBDT with calendar features."
        
    profile_records.append({
        'Drug': d,
        'ATC Description': drug_descriptions[d],
        'Trend': trend_desc,
        'Seasonality': seas_desc,
        'Stationarity': stat_desc,
        'Variance': var_desc,
        'ACF Lags': acf_desc,
        'Notes & Model Selection Bridge': notes
    })

profile_df = pd.DataFrame(profile_records)
print("Step 1: Computed Quantitative Drug Profiles.")

Step 1: Computed Quantitative Drug Profiles.


## Step 2: Display Master Drug Profile Matrix

In [5]:
# Export to CSV
export_dir = '.'
csv_export_path = 'drug_profiles_summary_table.csv'
profile_df.to_csv(csv_export_path, index=False)
print(f"Exported Drug Profile Table to: {os.path.abspath(csv_export_path)}\n")

pd.set_option('display.max_colwidth', None)
profile_df[['Drug', 'Trend', 'Seasonality', 'Stationarity', 'Notes & Model Selection Bridge']]
# profile_df[['Drug', 'Trend', 'Seasonality', 'Stationarity', 'Variance', 'ACF Lags', 'Notes & Model Selection Bridge']]

Exported Drug Profile Table to: c:\Users\ranje\sales forcasting\times_series\eda\drug_profiles_summary_table.csv



,Drug,Trend,Seasonality,Stationarity,Notes & Model Selection Bridge
0,M01AB,Strong Upward (+27.9%),Aug Peak (1.08x) vs Jun Min (0.94x) | Swing: 1.15x,Raw ADF p=0.0000 -> Diff ADF p=5.3e-27 (Stationary d=1),Noise-Dominated / Weak Seasonality. Regularized Ridge ML or LightGBM with short rolling features.
1,M01AE,Stable / Low Drift (-3.9%),Feb Peak (1.19x) vs Jun Min (0.91x) | Swing: 1.31x,Raw ADF p=0.0000 -> Diff ADF p=1.4e-25 (Stationary d=1),Noise-Dominated / Weak Seasonality. Regularized Ridge ML or LightGBM with short rolling features.
2,N02BA,Downward Drift (-46.8%),Feb Peak (1.20x) vs Sep Min (0.87x) | Swing: 1.39x,Raw ADF p=0.0000 -> Diff ADF p=2.3e-30 (Stationary d=1),Moderate Seasonality. SARIMAX or GBDT with calendar features.
3,N02BE,Downward Drift (-17.0%),Oct Peak (1.39x) vs Jul Min (0.65x) | Swing: 2.15x,Raw ADF p=0.0007 -> Diff ADF p=3.9e-29 (Stationary d=1),High-Volume / Winter Peak. Requires Log1p + LightGBM / Prophet for dual 7d/365d seasonality.
4,N05B,Downward Drift (-22.9%),Jan Peak (1.24x) vs May Min (0.84x) | Swing: 1.47x,Raw ADF p=0.0001 -> Diff ADF p=1.5e-27 (Stationary d=1),Consistent Downward Trend. Models require differencing (d=1) or explicit negative trend slope.
5,N05C,Downward Drift (-25.0%),Jan Peak (1.33x) vs Apr Min (0.80x) | Swing: 1.66x,Raw ADF p=0.0000 -> Diff ADF p=5.5e-27 (Stationary d=1),"Low Volume Intermittent Sales. Quantile Regression (P10, P50, P90) for zero-inflation safety stock."
6,R03,Stable / Low Drift (+12.7%),Dec Peak (1.42x) vs Jul Min (0.53x) | Swing: 2.68x,Raw ADF p=0.0000 -> Diff ADF p=4.6e-29 (Stationary d=1),High-Volume / Winter Peak. Requires Log1p + LightGBM / Prophet for dual 7d/365d seasonality.
7,R06,Strong Upward (+66.2%),May Peak (1.76x) vs Dec Min (0.48x) | Swing: 3.69x,Raw ADF p=0.0024 -> Diff ADF p=4.3e-29 (Stationary d=1),Massive Summer Allergy Surge (May/Jun). Requires Dedicated Per-Drug Model or TFT Embeddings.


## Step 3: Comparative Visualizations Across All 8 Drugs

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 14), dpi=150)
axes = axes.flatten()

for i, d in enumerate(drugs):
    m_avg = df.groupby('Month')[d].mean()
    m_norm = m_avg / m_avg.mean()
    
    axes[i].plot(m_norm.index, m_norm.values, marker='o', color='#1f77b4', linewidth=2)
    axes[i].axhline(1.0, color='red', linestyle='--', alpha=0.7)
    axes[i].set_title(f"{d} — {drug_descriptions[d]} (Seasonality Pattern)", fontweight='bold')
    axes[i].set_xlabel('Month of Year')
    axes[i].set_ylabel('Normalized Sales Ratio')
    axes[i].set_xticks(range(1, 13))
    axes[i].grid(True, linestyle='--', alpha=0.5)

plt.suptitle('Step 3: Normalized Monthly Seasonal Curves Across All 8 Drugs', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Step 4: Strategic Model Selection Summary

Based on the Drug Profile Matrix:

```
┌─────────────────────────────────────────────────────────────────────────────────────────────┐
│                             DRUG PROFILE TO MODEL MAPPING                                   │
├──────────────────────┬──────────────────────────────────────────┬───────────────────────────┤
│ Drug Profile Group   │ Dominant Empirical Characteristic        │ Target Recommended Model  │
├──────────────────────┼──────────────────────────────────────────┼───────────────────────────┤
│ M01AB, M01AE         │ Noise-dominant, Low Seasonality          │ Ridge / LightGBM (Short)  │
│ N02BA, N05B          │ Downward Drift, Linear Lags              │ SARIMAX (d=1) / Prophet   │
│ N02BE, R03           │ High Growth, Multiplicative Winter Peak  │ LightGBM Log1p + SHAP     │
│ R06                  │ Inverted Summer Allergy Surge            │ Per-Drug LightGBM / TFT   │
│ N05C                 │ Low-Volume Zero-Inflated Demand          │ Quantile GBDT (P10, P90)  │
└──────────────────────┴──────────────────────────────────────────┴───────────────────────────┘
```